In [0]:
# Import important libraries.
from pyspark.sql.functions import (
    col, explode, max as spark_max, current_timestamp, 
    row_number, expr, when, size, array
)
from pyspark.sql import Window

silver_table = "crimeworkspace.silver.silver_crime"
fact_table = "crimeworkspace.gold.fact_crime_incidents"
gold_schema = "crimeworkspace.gold"

In [0]:
## Handle incremental processing.

try:
    # Get the maximum ingest_ts already processed in fact table
    fact_df = spark.read.table(fact_table)
    max_processed_ts = fact_df.agg(spark_max("ingest_ts")).collect()[0][0]
    print(f"Watermark found: {max_processed_ts}")
    print(f"Processing only records after this timestamp...")
    
    # Read only NEW records from silver
    df_silver = (
        spark.read.table(silver_table)
        .filter(col("ingest_ts") > max_processed_ts)
    )
    
except Exception as e:
    # First run - process all data
    print("First run: Processing all silver data")
    df_silver = spark.read.table(silver_table)

record_count = df_silver.count()
print(f"Records to process: {record_count:,}")

if record_count == 0:
    print("No new records to process. Fact table is up to date!")
    dbutils.notebook.exit("No new data")

First run: Processing all silver data
Records to process: 192,708


# Explode Array Column (crime_codes)

In [0]:
# Explode the crime_codes array
# Step 1a: Handle NULL or empty arrays by replacing with crime_code
df_with_fallback = df_silver.withColumn(
    "crime_codes_cleaned",
    when(
        (col("crime_codes").isNull()) | (size(col("crime_codes")) == 0),
        array(col("crime_code"))  # Replace with single-element array containing crime_code
    ).otherwise(
        expr("filter(crime_codes, x -> x IS NOT NULL)")  # Filter out NULLs from array
    )
)

# Step 1b: Explode the cleaned array
df_exploded = df_with_fallback.select(
    "crime_report_id",
    "date_reported",
    "date_occurred",
    "time_occurred",
    "area_code",
    "area_name",
    "reporting_district",
    "latitude",
    "longitude",
    "premise_code",
    "premise_description",
    "premise_category",
    "crime_code",
    "crime_description",
    "crime_category",
    "crime_part",
    explode("crime_codes_cleaned").alias("exploded_crime_code"),
    "victim_age",
    "victim_sex",
    "victim_descent",
    "weapon_code",
    "weapon_description",
    "status_description",
    "ingest_ts",
    "source_file"
)
exploded_count = df_exploded.count()
print(f"Exploded to {exploded_count:,} rows")

Exploded to 209,273 rows


# Load Dimensions.

In [0]:
dim_date = spark.read.table(f"{gold_schema}.dim_date")
dim_time = spark.read.table(f"{gold_schema}.dim_time")
dim_police_station = spark.read.table(f"{gold_schema}.dim_police_station")
dim_location = spark.read.table(f"{gold_schema}.dim_location")
dim_crime_type = spark.read.table(f"{gold_schema}.dim_crime_type")
dim_victim = spark.read.table(f"{gold_schema}.dim_victim")
dim_weapon = spark.read.table(f"{gold_schema}.dim_weapon")

# Populate Fact Table.

In [0]:
# Join with Dim_Date (date_occurred)
fact = df_exploded.join(
    dim_date.select("date_key").withColumnRenamed("date_key", "date_occurred_key"),
    df_exploded.date_occurred == col("date_occurred_key"),
    "left"
)

# Join with Dim_Date (date_reported)
fact = fact.join(
    dim_date.select("date_key").withColumnRenamed("date_key", "date_reported_key"),
    fact.date_reported == col("date_reported_key"),
    "left"
)
# Join with Dim_Time
fact = fact.join(
    dim_time.select("time_key"),
    fact.time_occurred == dim_time.time_key,
    "left"
).withColumnRenamed("time_key", "time_key")

# Join with Dim_Police_Station
fact = fact.join(
    dim_police_station.select("station_key", "area_code", "reporting_district"),
    (fact.area_code == dim_police_station.area_code) &
    (fact.reporting_district == dim_police_station.reporting_district),
    "left"
)

# Join with Dim_Location
fact = fact.join(
    dim_location.select("location_key", "premise_code", "latitude", "longitude"),
    (fact.premise_code == dim_location.premise_code) &
    (fact.latitude == dim_location.latitude) &
    (fact.longitude == dim_location.longitude),
    "left"
)

# Join with Dim_Crime_Type
fact = fact.join(
    dim_crime_type.select("crime_type_key", "crime_code"),
    fact.crime_code == dim_crime_type.crime_code,
    "left"
)

# Join with Dim_Victim
fact = fact.join(
    dim_victim.select("victim_key", "victim_age", "victim_sex", "victim_descent"),
    (fact.victim_age == dim_victim.victim_age) &
    (fact.victim_sex == dim_victim.victim_sex) &
    (fact.victim_descent == dim_victim.victim_descent),
    "left"
)

# Join with Dim_Weapon
fact = fact.join(
    dim_weapon.select("weapon_key", "weapon_code"),
    fact.weapon_code == dim_weapon.weapon_code,
    "left"
)

# Select columns first
fact_pre = fact.select(
    # Business key
    col("crime_report_id"),
    
    # Foreign keys (surrogate keys from dimensions)
    col("date_occurred_key"),
    col("date_reported_key"),
    col("time_key"),
    col("station_key"),
    col("location_key"),
    col("crime_type_key"),
    col("victim_key"),
    col("weapon_key"),
    
    # Degenerate dimensions
    col("exploded_crime_code"),
    col("status_description"),
    
    # Metadata
    col("ingest_ts"),
    col("source_file"),
    current_timestamp().alias("fact_created_ts")
)

# Add incremental identity as primary key
# For incremental load, we need to get the max existing fact_key
try:
    existing_fact = spark.read.table(fact_table)
    max_fact_key = existing_fact.agg(spark_max("fact_key")).collect()[0][0]
    if max_fact_key is None:
        max_fact_key = 0
    print(f"Starting fact_key from: {max_fact_key + 1}")
except:
    max_fact_key = 0
    print("Starting fact_key from: 1")

# Generate identity key starting from max_fact_key + 1
window_spec = Window.orderBy("crime_report_id", "exploded_crime_code")
fact_final = fact_pre.withColumn(
    "fact_key",
    row_number().over(window_spec) + max_fact_key
)

# Reorder columns with fact_key as first column
fact_final = fact_final.select(
    "fact_key",
    "crime_report_id",
    "date_occurred_key",
    "date_reported_key",
    "time_key",
    "station_key",
    "location_key",
    "crime_type_key",
    "victim_key",
    "weapon_key",
    "exploded_crime_code",
    "status_description",
    "ingest_ts",
    "source_file",
    "fact_created_ts"
)

Starting fact_key from: 1


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
final_count = fact_final.count()

fact_final.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable(fact_table)

print(f"{final_count:,} records written to {fact_table}")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


209,273 records written to crimeworkspace.gold.fact_crime_incidents


In [0]:
%sql
SELECT * FROM crimeworkspace.gold.fact_crime_incidents

fact_key,crime_report_id,date_occurred_key,date_reported_key,time_key,station_key,location_key,crime_type_key,victim_key,weapon_key,exploded_crime_code,status_description,ingest_ts,source_file,fact_created_ts
1,817,2020-09-19,2020-09-20,17:00,934,23770,55,22,1,510.0,Investigation Continued,2025-11-09T15:15:20.711Z,/Volumes/crimeworkspace/default/raw/crime_2020.csv,2025-11-10T21:49:52.947Z
2,10304468,2020-01-08,2020-01-08,22:30,146,74776,60,683,61,624.0,Adult Other,2025-11-09T15:15:20.711Z,/Volumes/crimeworkspace/default/raw/crime_2020.csv,2025-11-10T21:49:52.947Z
3,190101086,2020-01-01,2020-01-02,03:30,41,28919,60,388,62,624.0,Investigation Continued,2025-11-09T15:15:20.711Z,/Volumes/crimeworkspace/default/raw/crime_2020.csv,2025-11-10T21:49:52.947Z
4,190101087,2020-01-01,2020-01-02,05:10,35,92394,62,1099,61,626.0,Adult Arrested,2025-11-09T15:15:20.711Z,/Volumes/crimeworkspace/default/raw/crime_2020.csv,2025-11-10T21:49:52.947Z
5,190326475,2020-03-01,2020-03-01,21:30,362,10095,55,12,1,510.0,Adult Arrested,2025-11-09T15:15:20.711Z,/Volumes/crimeworkspace/default/raw/crime_2020.csv,2025-11-10T21:49:52.947Z
6,190326475,2020-03-01,2020-03-01,21:30,362,10095,55,12,1,998.0,Adult Arrested,2025-11-09T15:15:20.711Z,/Volumes/crimeworkspace/default/raw/crime_2020.csv,2025-11-10T21:49:52.947Z
7,191501505,2020-01-01,2020-01-01,17:30,790,96133,79,1556,1,745.0,Investigation Continued,2025-11-09T15:15:20.711Z,/Volumes/crimeworkspace/default/raw/crime_2020.csv,2025-11-10T21:49:52.947Z
8,191501505,2020-01-01,2020-01-01,17:30,790,96133,79,1556,1,998.0,Investigation Continued,2025-11-09T15:15:20.711Z,/Volumes/crimeworkspace/default/raw/crime_2020.csv,2025-11-10T21:49:52.947Z
9,191921269,2020-01-01,2020-01-01,04:15,1048,69265,78,568,1,740.0,Investigation Continued,2025-11-09T15:15:20.711Z,/Volumes/crimeworkspace/default/raw/crime_2020.csv,2025-11-10T21:49:52.947Z
10,200100001,2020-02-25,2020-02-26,20:00,4,40334,55,22,1,510.0,Adult Arrested,2025-11-09T15:15:20.711Z,/Volumes/crimeworkspace/default/raw/crime_2020.csv,2025-11-10T21:49:52.947Z


In [0]:
# Verify join integrity (check for null keys)
print("\n🔍 Join integrity check:")
spark.sql(f"""
    SELECT 
        SUM(CASE WHEN date_occurred_key IS NULL THEN 1 ELSE 0 END) as null_date_occurred,
        SUM(CASE WHEN time_key IS NULL THEN 1 ELSE 0 END) as null_time,
        SUM(CASE WHEN station_key IS NULL THEN 1 ELSE 0 END) as null_station,
        SUM(CASE WHEN location_key IS NULL THEN 1 ELSE 0 END) as null_location,
        SUM(CASE WHEN crime_type_key IS NULL THEN 1 ELSE 0 END) as null_crime_type,
        SUM(CASE WHEN victim_key IS NULL THEN 1 ELSE 0 END) as null_victim,
        SUM(CASE WHEN weapon_key IS NULL THEN 1 ELSE 0 END) as null_weapon
    FROM {fact_table}
    """).show()


🔍 Join integrity check:
+------------------+---------+------------+-------------+---------------+-----------+-----------+
|null_date_occurred|null_time|null_station|null_location|null_crime_type|null_victim|null_weapon|
+------------------+---------+------------+-------------+---------------+-----------+-----------+
|                 0|        0|           0|            0|              0|          0|          0|
+------------------+---------+------------+-------------+---------------+-----------+-----------+

